# Phase 8 — Le Conseil a lu trois relevés

## Objectifs

- Construire la liste des mots interdits : les formes retenues, leurs variantes d'écriture, leurs
  pluriels, et les doublons produits par la fusion de la phase 3.
- Interdire ce vocabulaire au texte, à l'apprentissage comme à l'évaluation, et **prouver** que
  l'interdiction est effective (un compte à zéro, calculé par le code).
- Réentraîner le modèle de référence à l'identique sur le texte expurgé et rendre la chute, globale
  et par classe.
- Reprendre le modèle `EmbeddingBag + MLP` de la phase 3 — la référence défendue depuis le début de cet
  acte — plutôt que la variante convolutive expérimentale des phases 6-7, pour une comparaison rapide
  et directement lisible.


## 1. Imports

In [1]:
from pathlib import Path
import csv
import random
import re
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset


## 2. Configuration et reproductibilité (identique à la phase 3)

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

URL_DATA = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/"
    "csv-data/ufo-complete-geocoded-time-standardized.csv"
)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE8_DIR = OUTPUT_DIR / "phase_8_vocabulaire_interdit"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PHASE8_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

TEST_SIZE = 0.20
SEUIL_MIN_CLASSE = 5
BATCH_SIZE = 128
EMBEDDING_DIM = 96
HIDDEN_DIM = 128
DROPOUT = 0.30
LEARNING_RATE = 0.003
WEIGHT_DECAY = 0.0001
N_EPOCHS = 20
PATIENCE = 4


## 3. Téléchargement, préparation et découpe (identiques à la phase 3)

In [3]:
if not DATA_PATH.exists():
    print("Téléchargement du fichier...")
    urllib.request.urlretrieve(URL_DATA, DATA_PATH)
else:
    print(f"Fichier déjà disponible : {DATA_PATH}")

lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()
df["shape_model"] = df["shape_clean"].replace({"round": "circle", "changed": "changing"})

masque_forme_manquante = df["shape_clean"].eq("")
masque_fourre_tout = df["shape_model"].isin(["unknown", "other"])
masque_commentaire_vide = df["comments_clean"].eq("")

df_avant_filtre_classes_rares = df.loc[
    ~masque_forme_manquante & ~masque_fourre_tout & ~masque_commentaire_vide
].copy()

compte_classes = df_avant_filtre_classes_rares["shape_model"].value_counts()
classes_conservees = compte_classes.loc[compte_classes >= SEUIL_MIN_CLASSE].index

df_modele = df_avant_filtre_classes_rares.loc[
    df_avant_filtre_classes_rares["shape_model"].isin(classes_conservees)
].copy()

X = df_modele["comments_clean"].copy()
y = df_modele["shape_model"].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y,
)

print(f"Relevés gardés : {len(df_modele)} | train : {len(X_train)} | validation : {len(X_val)}")
print(f"Classes retenues ({y.nunique()}) : {sorted(y.unique())}")


Fichier déjà disponible : ..\data\releves_klaxo3.csv
Relevés gardés : 73177 | train : 58541 | validation : 14636
Classes retenues (20) : ['changing', 'chevron', 'cigar', 'circle', 'cone', 'cross', 'cylinder', 'delta', 'diamond', 'disk', 'egg', 'fireball', 'flash', 'formation', 'light', 'oval', 'rectangle', 'sphere', 'teardrop', 'triangle']


## 4. La liste des mots interdits

Trois sources, combinées : (1) les 20 formes retenues par le modèle, (2) les deux doublons que la
fusion de la phase 3 a produits (`round` → `circle`, `changed` → `changing`) — ce sont des mots réels
qui décrivent la forme et qui n'apparaissent plus tels quels comme nom de classe, mais qui restent des
indices directs, (3) le pluriel de chaque mot ci-dessus, et une variante d'écriture connue
(`disk`/`disc`, orthographe britannique).

In [4]:
def pluriel(mot):
    if mot.endswith(("s", "x", "ch", "sh")):
        return mot + "es"
    if mot.endswith("y") and mot[-2] not in "aeiou":
        return mot[:-1] + "ies"
    return mot + "s"

formes_retenues = sorted(y.unique())
doublons_fusionnes = ["round", "changed"]
variantes_ecriture = {"disk": ["disc"]}

mots_de_base = list(formes_retenues) + doublons_fusionnes
mots_interdits = set()
for mot in mots_de_base:
    mots_interdits.add(mot)
    mots_interdits.add(pluriel(mot))
    for variante in variantes_ecriture.get(mot, []):
        mots_interdits.add(variante)
        mots_interdits.add(pluriel(variante))

mots_interdits = sorted(mots_interdits)
print(f"Nombre de mots interdits : {len(mots_interdits)}")
print(mots_interdits)


Nombre de mots interdits : 46
['changed', 'changeds', 'changing', 'changings', 'chevron', 'chevrons', 'cigar', 'cigars', 'circle', 'circles', 'cone', 'cones', 'cross', 'crosses', 'cylinder', 'cylinders', 'delta', 'deltas', 'diamond', 'diamonds', 'disc', 'discs', 'disk', 'disks', 'egg', 'eggs', 'fireball', 'fireballs', 'flash', 'flashes', 'formation', 'formations', 'light', 'lights', 'oval', 'ovals', 'rectangle', 'rectangles', 'round', 'rounds', 'sphere', 'spheres', 'teardrop', 'teardrops', 'triangle', 'triangles']


## 5. Statistiques avant interdiction : le mot de la forme est-il dans le texte ?

Le Conseil cite deux chiffres : 34,7 % des relevés contiennent tel quel le mot de leur propre forme,
72,6 % pour la forme `light`, et l'analyste ajoute 9,9 % pour `circle`. On recalcule ces trois chiffres
sur nos propres données, comme repère de cohérence avant de construire quoi que ce soit.

In [5]:
def contient_le_mot_de_sa_forme(ligne):
    mot = ligne["shape_model"]
    variantes = {mot, pluriel(mot)}
    if mot == "circle":
        variantes |= {"round", pluriel("round")}
    if mot == "changing":
        variantes |= {"changed", pluriel("changed")}
    tokens = set(re.findall(r"[a-z0-9]+", ligne["comments_clean"].lower()))
    return len(tokens & variantes) > 0

df_modele["contient_mot_de_forme"] = df_modele.apply(contient_le_mot_de_sa_forme, axis=1)

part_globale = df_modele["contient_mot_de_forme"].mean()
part_light = df_modele.loc[df_modele["shape_model"] == "light", "contient_mot_de_forme"].mean()
part_circle = df_modele.loc[df_modele["shape_model"] == "circle", "contient_mot_de_forme"].mean()

print(f"Part globale (nos données) : {part_globale:.1%}  (Conseil : 34.7%)")
print(f"Part pour 'light'          : {part_light:.1%}  (Conseil : 72.6%)")
print(f"Part pour 'circle'         : {part_circle:.1%}  (Conseil : 9.9%)")


Part globale (nos données) : 40.5%  (Conseil : 34.7%)
Part pour 'light'          : 71.4%  (Conseil : 72.6%)
Part pour 'circle'         : 19.1%  (Conseil : 9.9%)


## 6. Application de l'interdiction et vérification

Un premier essai a supprimé les mots interdits par une regex à bordures de mot (`\bmot\b`) sur le
texte brut. Elle a laissé passer un relevé : `"...ball of light_w/aura shoots..."`, où le caractère
`_` compte comme caractère de mot pour `\b` — `light_w` n'a donc jamais de frontière avant `w`, et
`light` n'est jamais isolé aux yeux de la regex. Mais le tokenizer utilisé partout ailleurs dans ce
projet (`[a-z0-9]+`) ne connaît pas `_` : lui coupe bien `light_w` en `light` et `w`. Les deux
définitions de « mot » n'étaient pas les mêmes.

La correction retenue : ne plus jamais raisonner sur le texte brut pour cette interdiction, mais sur
la **même tokenisation** que celle utilisée pour construire le vocabulaire du modèle. On retire les
jetons interdits de la liste de jetons, puis on rejoint — ce qui garantit que ce qui est « interdit »
et ce que « voit » le modèle sont exactement définis de la même façon.

In [6]:
def tokenizer(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

ensemble_mots_interdits = set(mots_interdits)

def expurger(texte):
    tokens = tokenizer(texte)
    tokens_gardes = [t for t in tokens if t not in ensemble_mots_interdits]
    return " ".join(tokens_gardes)

df_modele["comments_sans_forme"] = df_modele["comments_clean"].apply(expurger)

def compte_mots_interdits_restants(serie_textes):
    compte = 0
    for texte in serie_textes:
        tokens = set(tokenizer(texte))
        if tokens & ensemble_mots_interdits:
            compte += 1
    return compte

compte_avant = compte_mots_interdits_restants(df_modele["comments_clean"])
compte_apres = compte_mots_interdits_restants(df_modele["comments_sans_forme"])

print(f"Relevés contenant un mot interdit AVANT expurgation : {compte_avant}")
print(f"Relevés contenant un mot interdit APRÈS expurgation : {compte_apres}")
assert compte_apres == 0, "L'interdiction n'est pas effective : des mots interdits subsistent."
print("Interdiction vérifiée effective (compte = 0).")


Relevés contenant un mot interdit AVANT expurgation : 45232
Relevés contenant un mot interdit APRÈS expurgation : 0
Interdiction vérifiée effective (compte = 0).


## 7. Découpe train/validation (identique à la phase 3), texte original et texte expurgé

In [7]:
X_original = df_modele["comments_clean"].copy()
X_expurge = df_modele["comments_sans_forme"].copy()
y_cible = df_modele["shape_model"].copy()

# La meme decoupe (meme graine, meme stratification) est appliquee aux deux versions du texte,
# en repartant des memes index pour garantir que "avant" et "apres" partagent train/val identiques.
index_train, index_val = train_test_split(
    df_modele.index, test_size=TEST_SIZE, random_state=SEED, stratify=y_cible,
)

X_train_original, X_val_original = X_original.loc[index_train], X_original.loc[index_val]
X_train_expurge, X_val_expurge = X_expurge.loc[index_train], X_expurge.loc[index_val]
y_train, y_val = y_cible.loc[index_train], y_cible.loc[index_val]

print(f"Train : {len(index_train)} | Validation : {len(index_val)}")


Train : 58541 | Validation : 14636


## 8. Vocabulaire, jeu de données, modèle et boucle d'entraînement (identiques à la phase 3)

In [8]:
def tokenizer(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

class DatasetTextes(Dataset):
    def __init__(self, textes, labels, vocabulaire):
        self.textes = list(textes)
        self.labels = list(labels)
        self.vocabulaire = vocabulaire

    def __len__(self):
        return len(self.textes)

    def __getitem__(self, index):
        tokens = tokenizer(self.textes[index])
        ids = [self.vocabulaire.get(t, self.vocabulaire["<UNK>"]) for t in tokens]
        if len(ids) == 0:
            ids = [self.vocabulaire["<UNK>"]]
        return torch.tensor(ids, dtype=torch.long), int(self.labels[index])

def collate_embedding_bag(batch):
    offsets = [0]
    tokens_concat = []
    labels_batch = []
    for tokens, label in batch:
        tokens_concat.extend(tokens.tolist())
        labels_batch.append(label)
        offsets.append(offsets[-1] + len(tokens))
    return (
        torch.tensor(tokens_concat, dtype=torch.long),
        torch.tensor(offsets[:-1], dtype=torch.long),
        torch.tensor(labels_batch, dtype=torch.long),
    )

class ClassifieurPyTorch(nn.Module):
    def __init__(self, taille_vocabulaire, nombre_classes):
        super().__init__()
        self.embedding = nn.EmbeddingBag(taille_vocabulaire, EMBEDDING_DIM, mode="mean")
        self.reseau = nn.Sequential(
            nn.Linear(EMBEDDING_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, nombre_classes),
        )

    def forward(self, tokens, offsets):
        return self.reseau(self.embedding(tokens, offsets))

def une_epoque_train(modele, loader, optimiseur, fonction_perte):
    modele.train()
    perte_totale, nombre_exemples = 0.0, 0
    for tokens, offsets, labels in loader:
        optimiseur.zero_grad()
        logits = modele(tokens, offsets)
        perte = fonction_perte(logits, labels)
        perte.backward()
        optimiseur.step()
        perte_totale += perte.item() * len(labels)
        nombre_exemples += len(labels)
    return perte_totale / nombre_exemples

def evaluer_modele(modele, loader, fonction_perte):
    modele.eval()
    perte_totale, nombre_exemples = 0.0, 0
    predictions, labels_reels = [], []
    with torch.no_grad():
        for tokens, offsets, labels in loader:
            logits = modele(tokens, offsets)
            perte = fonction_perte(logits, labels)
            perte_totale += perte.item() * len(labels)
            nombre_exemples += len(labels)
            predictions.extend(logits.argmax(dim=1).tolist())
            labels_reels.extend(labels.tolist())
    accuracy = accuracy_score(labels_reels, predictions)
    return perte_totale / nombre_exemples, accuracy, labels_reels, predictions

def entrainer_configuration(nom, X_train_texte, X_val_texte, y_train, y_val):
    vocabulaire = {"<PAD>": 0, "<UNK>": 1}
    for texte in X_train_texte:
        for token in tokenizer(texte):
            if token not in vocabulaire:
                vocabulaire[token] = len(vocabulaire)

    label_encoder = LabelEncoder()
    y_train_ids = label_encoder.fit_transform(y_train)
    y_val_ids = label_encoder.transform(y_val)

    dataset_train = DatasetTextes(X_train_texte, y_train_ids, vocabulaire)
    dataset_val = DatasetTextes(X_val_texte, y_val_ids, vocabulaire)
    loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_embedding_bag)
    loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_embedding_bag)

    torch.manual_seed(SEED)
    modele = ClassifieurPyTorch(len(vocabulaire), len(label_encoder.classes_)).to(DEVICE)
    optimiseur = torch.optim.AdamW(modele.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    fonction_perte = nn.CrossEntropyLoss()

    meilleure_perte_val = float("inf")
    meilleur_etat = None
    epochs_sans_amelioration = 0
    debut = time.perf_counter()

    for epoch in range(1, N_EPOCHS + 1):
        perte_train = une_epoque_train(modele, loader_train, optimiseur, fonction_perte)
        perte_val, acc_val, _, _ = evaluer_modele(modele, loader_val, fonction_perte)
        print(f"[{nom}] epoch {epoch:02d} | train={perte_train:.4f} | val={perte_val:.4f} | acc={acc_val:.2%}")

        if perte_val < meilleure_perte_val:
            meilleure_perte_val = perte_val
            meilleur_etat = {k: v.cpu().clone() for k, v in modele.state_dict().items()}
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= PATIENCE:
            print(f"[{nom}] arrêt anticipé à l'époque {epoch}.")
            break

    temps_total = time.perf_counter() - debut
    modele.load_state_dict(meilleur_etat)

    perte_finale, accuracy_finale, labels_reels, predictions_ids = evaluer_modele(modele, loader_val, fonction_perte)
    predictions = label_encoder.inverse_transform(predictions_ids)
    y_val_texte = label_encoder.inverse_transform(labels_reels)

    rapport = classification_report(y_val_texte, predictions, output_dict=True, zero_division=0)

    return {
        "nom": nom, "accuracy": accuracy_finale, "temps": temps_total,
        "rapport_classes": rapport, "taille_vocabulaire": len(vocabulaire),
    }


## 9. Entraînement AVANT interdiction (texte original, référence de la phase 3)

In [9]:
resultat_avant = entrainer_configuration("avant_interdiction", X_train_original, X_val_original, y_train, y_val)


[avant_interdiction] epoch 01 | train=2.0155 | val=1.7910 | acc=49.23%
[avant_interdiction] epoch 02 | train=1.6905 | val=1.7082 | acc=52.75%
[avant_interdiction] epoch 03 | train=1.5417 | val=1.6935 | acc=53.14%
[avant_interdiction] epoch 04 | train=1.4234 | val=1.7122 | acc=53.55%
[avant_interdiction] epoch 05 | train=1.3163 | val=1.7547 | acc=53.59%
[avant_interdiction] epoch 06 | train=1.2159 | val=1.8239 | acc=53.42%
[avant_interdiction] epoch 07 | train=1.1254 | val=1.9290 | acc=52.71%
[avant_interdiction] arrêt anticipé à l'époque 7.


## 10. Entraînement APRÈS interdiction (texte expurgé du vocabulaire des formes)

In [10]:
resultat_apres = entrainer_configuration("apres_interdiction", X_train_expurge, X_val_expurge, y_train, y_val)


[apres_interdiction] epoch 01 | train=2.2823 | val=2.1588 | acc=34.58%
[apres_interdiction] epoch 02 | train=2.0895 | val=2.1083 | acc=36.25%
[apres_interdiction] epoch 03 | train=1.9757 | val=2.1061 | acc=37.07%
[apres_interdiction] epoch 04 | train=1.8679 | val=2.1276 | acc=37.11%
[apres_interdiction] epoch 05 | train=1.7652 | val=2.1802 | acc=37.52%
[apres_interdiction] epoch 06 | train=1.6649 | val=2.2511 | acc=36.18%
[apres_interdiction] epoch 07 | train=1.5684 | val=2.3445 | acc=35.52%
[apres_interdiction] arrêt anticipé à l'époque 7.


## 11. La chute, globale et par classe

In [11]:
print(f"Accuracy avant interdiction : {resultat_avant['accuracy']:.2%}")
print(f"Accuracy après interdiction  : {resultat_apres['accuracy']:.2%}")
print(f"Chute (accuracy globale)     : {resultat_apres['accuracy'] - resultat_avant['accuracy']:+.2%}")

rapport_avant = pd.DataFrame(resultat_avant["rapport_classes"]).transpose()
rapport_apres = pd.DataFrame(resultat_apres["rapport_classes"]).transpose()

macro_f1_avant = resultat_avant["rapport_classes"]["macro avg"]["f1-score"]
macro_f1_apres = resultat_apres["rapport_classes"]["macro avg"]["f1-score"]

print(f"\nAccuracy globale (dominée par les classes fréquentes)  : {resultat_avant['accuracy']:.2%} -> {resultat_apres['accuracy']:.2%} "
      f"({resultat_apres['accuracy'] - resultat_avant['accuracy']:+.2%})")
print(f"F1 macro-moyenné (chaque classe pèse pareil)            : {macro_f1_avant:.2%} -> {macro_f1_apres:.2%} "
      f"({macro_f1_apres - macro_f1_avant:+.2%})")


Accuracy avant interdiction : 53.14%
Accuracy après interdiction  : 37.07%
Chute (accuracy globale)     : -16.07%

Accuracy globale (dominée par les classes fréquentes)  : 53.14% -> 37.07% (-16.07%)
F1 macro-moyenné (chaque classe pèse pareil)            : 42.47% -> 15.09% (-27.39%)


In [12]:
comparaison_classes = pd.DataFrame({
    "f1_avant": rapport_avant["f1-score"],
    "f1_apres": rapport_apres["f1-score"],
}).drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
comparaison_classes["chute_f1"] = comparaison_classes["f1_apres"] - comparaison_classes["f1_avant"]
comparaison_classes = comparaison_classes.sort_values("chute_f1")

print("Classes qui s'effondrent le plus (F1 avant -> après) :")
comparaison_classes.head(5)


Classes qui s'effondrent le plus (F1 avant -> après) :


,f1_avant,f1_apres,chute_f1
diamond,0.565947,0.000000,-0.565947
cigar,0.602817,0.120000,-0.482817
egg,0.460870,0.000000,-0.460870
chevron,0.460481,0.028846,-0.431635
teardrop,0.396135,0.000000,-0.396135


## 12. Interprétation : deux résumés, deux histoires

L'accuracy globale est pondérée par la fréquence des classes : `light`, `triangle` et `circle` pèsent
lourd dans le total, donc leur comportement domine le chiffre global. Le F1 macro-moyenné donne le
même poids à chaque classe, y compris les plus rares : une classe qui s'effondre complètement compte
alors autant qu'une classe qui ne bouge pas. Si le F1 macro chute davantage que l'accuracy globale,
c'est le signe qu'une partie du score de la phase 3 provenait de la reconnaissance de quelques
classes (souvent les plus fréquentes, où le mot de la forme apparaît le plus souvent dans le texte —
72,6 % pour `light`) plutôt que d'une compréhension réellement répartie sur les 20 formes.

## 13. Export des résultats

In [13]:
mots_interdits_df = pd.DataFrame({"mot_interdit": mots_interdits})
mots_interdits_df.to_csv(PHASE8_DIR / "liste_mots_interdits.csv", index=False)

rapport_avant.to_csv(PHASE8_DIR / "scores_par_classe_avant.csv")
rapport_apres.to_csv(PHASE8_DIR / "scores_par_classe_apres.csv")
comparaison_classes.to_csv(PHASE8_DIR / "comparaison_classes.csv")

resume_phase8 = pd.DataFrame([{
    "nombre_mots_interdits": len(mots_interdits),
    "part_relevés_mot_present_avant_global": part_globale,
    "part_relevés_mot_present_avant_light": part_light,
    "part_relevés_mot_present_avant_circle": part_circle,
    "compte_mots_interdits_restants_apres": compte_apres,
    "accuracy_avant": resultat_avant["accuracy"],
    "accuracy_apres": resultat_apres["accuracy"],
    "chute_accuracy": resultat_apres["accuracy"] - resultat_avant["accuracy"],
    "macro_f1_avant": macro_f1_avant,
    "macro_f1_apres": macro_f1_apres,
    "chute_macro_f1": macro_f1_apres - macro_f1_avant,
    "classes_qui_s_effondrent_le_plus": ", ".join(comparaison_classes.head(3).index.tolist()),
}])
resume_phase8.to_csv(PHASE8_DIR / "resume_phase8.csv", index=False)

resume_phase8


,nombre_mots_interdits,part_relevés_mot_present_avant_global,part_relevés_mot_present_avant_light,part_relevés_mot_present_avant_circle,compte_mots_interdits_restants_apres,accuracy_avant,accuracy_apres,chute_accuracy,macro_f1_avant,macro_f1_apres,chute_macro_f1,classes_qui_s_effondrent_le_plus
0,46,0.405346,0.714214,0.191152,0,0.531429,0.37073,-0.1607,0.424749,0.150881,-0.273869,"diamond, cigar, egg"
